# Multimodal RAG - Google Colab Setup

This notebook sets up the existing Multimodal Offline RAG project in Google Colab with GPU acceleration.

It mounts Google Drive, installs dependencies, verifies the GPU, configures the app, starts the FastAPI backend, and exposes it through a public tunnel for browser access.

Use this notebook on a Google Colab runtime with a GPU enabled.

In [ ]:
# Cell 1: check the runtime environment
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU count:', torch.cuda.device_count())
else:
    print('No GPU detected. Switch runtime to a GPU-enabled Colab instance.')

In [ ]:
# Cell 2: mount Google Drive for persistent data
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/MultimodalRag'
!mkdir -p "$PROJECT_ROOT"
print('Drive root:', PROJECT_ROOT)

In [ ]:
# Cell 3: clone or prepare the repository
import os
os.chdir('/content')

# If the repo is not already in Drive, clone it from GitHub.
REPO_URL = 'https://github.com/Govind252005/Rag.git'
REPO_DIR = '/content/project'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repository already exists at', REPO_DIR)

!ls -la /content
!ls -la /content/project | head

In [ ]:
# Cell 4: install the required system dependencies
!apt-get update
!apt-get install -y tesseract-ocr
!python -m pip install --upgrade pip setuptools wheel

In [ ]:
# Cell 5: install Python dependencies in the correct order
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install sentence-transformers==3.3.1
!pip install transformers==4.47.1
!pip install chromadb
!pip install fastapi==0.115.6 uvicorn[standard]==0.34.0 python-multipart==0.0.20
!pip install "pydantic>=2.12.5" "pydantic-settings>=2.7.1"
!pip install paddleocr==3.7.0 pytesseract==0.3.13 pdf2image==1.17.0
!pip install faster-whisper==1.1.0
!pip install PyMuPDF==1.25.1 python-docx==1.1.2
!pip install "ollama>=0.5.1"
!pip install "google-genai>=0.3.0" "openai>=1.0.0" "anthropic>=0.30.0" "groq>=0.9.0"
!pip install "cryptography>=42.0.0"
!pip install numpy==1.26.4 python-dotenv==1.0.1 rank-bm25==0.2.2 redis==5.0.8

In [ ]:
# Cell 6: configure project paths and runtime variables
import os

os.environ['RAG_DATA_HOME'] = '/content/drive/MyDrive/MultimodalRag'
os.environ['OLLAMA_HOST'] = 'http://localhost:11434'
os.environ['LLM_MODEL'] = 'qwen3:8b'
os.environ['DEFAULT_PROVIDER'] = 'ollama'
os.environ['OCR_ENGINE'] = 'paddleocr'

print('RAG_DATA_HOME=', os.environ.get('RAG_DATA_HOME'))
print('OLLAMA_HOST=', os.environ.get('OLLAMA_HOST'))
print('LLM_MODEL=', os.environ.get('LLM_MODEL'))

In [ ]:
# Cell 7: create persistent folders for data, storage, logs, and models
!mkdir -p /content/drive/MyDrive/MultimodalRag/data
!mkdir -p /content/drive/MyDrive/MultimodalRag/storage/chroma
!mkdir -p /content/drive/MyDrive/MultimodalRag/models
!mkdir -p /content/drive/MyDrive/MultimodalRag/logs
!ls -R /content/drive/MyDrive/MultimodalRag | head -n 50

In [ ]:
# Cell 8: verify backend can import config and check GPU settings
import sys
sys.path.insert(0, '/content/project/backend')

import config
print('DEVICE:', config.DEVICE)
print('DATA_DIR:', config.DATA_DIR)
print('STORE_DIR:', config.STORE_DIR)
print('CHROMA_DIR:', config.CHROMA_DIR)
print('OLLAMA_HOST:', config.OLLAMA_HOST)
print('LLM_MODEL:', config.LLM_MODEL)

In [ ]:
# Cell 9: start the FastAPI backend in the background
import subprocess, os, time, signal

backend_dir = '/content/project/backend'
backend_log = open('/content/drive/MyDrive/MultimodalRag/logs/backend.log', 'ab')

proc = subprocess.Popen(
    ['uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd=backend_dir,
    stdout=backend_log,
    stderr=subprocess.STDOUT,
)
print('Backend PID:', proc.pid)
time.sleep(8)
print('Checking if backend is responding...')
!curl -s http://127.0.0.1:8000/api/health || true

In [ ]:
# Cell 10: expose the backend with ngrok
!pip install pyngrok
from pyngrok import ngrok

# If a tunnel already exists, reuse it.
tunnel = None
for candidate in ngrok.get_tunnels():
    if candidate.public_url.startswith('http://') or candidate.public_url.startswith('https://'):
        tunnel = candidate
        break

if tunnel is None:
    tunnel = ngrok.connect(8000, 'http')

print('Public URL:', tunnel.public_url)

## Frontend usage

The frontend in this repo expects to call the backend via `/api` and usually works best with a local Vite dev server pointing to the public Colab backend URL.

You can either:
- keep the frontend local on your machine and point it to the public Colab backend URL, or
- run the frontend in Colab too on port 5173.

### Optional: run frontend in Colab

```bash
cd /content/project/frontend
npm install
npm run dev -- --host 0.0.0.0 --port 5173
```

Then expose port 5173 with another tunnel if needed.

In [ ]:
# Cell 11: optional frontend launch in Colab
# Uncomment this block if you want to run the Vite frontend on the Colab runtime itself.
# !cd /content/project/frontend && npm install && npm run dev -- --host 0.0.0.0 --port 5173

## Test the backend

Run this to validate that the API is live and that the project is accessible from the public URL.

```python
import requests
r = requests.get('http://127.0.0.1:8000/api/health')
print(r.status_code)
print(r.json())
```

For an upload-based test, open the Swagger UI at the public URL and use `POST /api/ingest` with a document.